# Chapter 11. 파이썬 클래스와 객체

이 노트북은 [Chapter 11 원본 문서](../doc/Chapter%2011.%20%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%ED%81%B0%EB%9E%98%EC%8A%A4%EC%99%80%20%EA%B0%9D%EC%B2%B4.md)를 실습용으로 변환한 자료입니다.

이름, 객체, 스코프, 클래스, 상속, 데이터클래스, 반복자, 제너레이터까지 코인 거래 예제로 연습합니다.

## 1. 이름과 객체

변수는 데이터를 담는 상자가 아니라, 객체에 붙은 이름입니다. 같은 객체를 여러 이름으로 참조할 수 있습니다.

In [ ]:
prices = [100, 105, 110]
alias = prices

alias.append(115)
print("prices:", prices)
print("alias:", alias)
print("같은 객체인가?", prices is alias)

copied_prices = prices.copy()
copied_prices.append(120)
print("원본:", prices)
print("복사본:", copied_prices)

## 2. 스코프와 네임스페이스

네임스페이스는 이름과 객체를 연결하는 저장 공간입니다. 함수 안의 지역 변수와 전역 변수는 서로 독립적입니다.

In [ ]:
message = "전역 메시지"


def show_message():
    message = "지역 메시지"
    print("함수 내부:", message)


show_message()
print("함수 외부:", message)


def make_counter():
    count = 0

    def increase():
        nonlocal count
        count += 1
        return count

    return increase

counter = make_counter()
print(counter())
print(counter())

## 3. 클래스 정의하기

클래스는 관련된 데이터와 동작을 하나의 타입으로 묶어줍니다.

In [ ]:
class Coin:
    """코인 정보를 표현하는 클래스입니다."""

    pass


bitcoin = Coin()
print(type(bitcoin))
print(bitcoin.__class__)


class Market:
    name = "가상 거래소"

    def describe(self):
        return f"거래소: {self.name}"


market = Market()
print(market.describe())
print(Market.name)

## 4. `__init__()`와 인스턴스 속성

인스턴스마다 다른 초기값을 넣고 싶다면 `__init__()`를 정의합니다.

In [ ]:
class Coin:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self.price = price

    def display(self):
        return f"{self.symbol}: {self.price:,}원"


bitcoin = Coin("BTC", 105_000_000)
ethereum = Coin("ETH", 3_500_000)
print(bitcoin.display())
print(ethereum.display())

## 5. 인스턴스 메서드와 상태 변경

메서드는 객체의 상태를 읽고 바꾸는 로직을 담는 함수입니다.

In [ ]:
class Wallet:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("잔액이 부족합니다.")
        self.balance -= amount

    def summary(self):
        return f"잔액: {self.balance:,}원"


wallet = Wallet(100_000)
wallet.deposit(50_000)
wallet.withdraw(30_000)
print(wallet.summary())

## 6. 클래스 변수와 인스턴스 변수

인스턴스 변수는 각 객체마다 별도로 저장되고, 클래스 변수는 모든 객체가 공유합니다.

In [ ]:
class Order:
    market_type = "spot"

    def __init__(self, symbol, amount):
        self.symbol = symbol
        self.amount = amount


first_order = Order("BTC", 10_000)
second_order = Order("ETH", 20_000)
print(first_order.market_type)
print(second_order.market_type)
print(first_order.symbol, second_order.symbol)


class Portfolio:
    def __init__(self):
        self.symbols = []

    def add(self, symbol):
        self.symbols.append(symbol)


first = Portfolio()
second = Portfolio()
first.add("BTC")
print(first.symbols)
print(second.symbols)

## 7. 속성 검색과 메서드 객체

속성 조회는 인스턴스 → 클래스 순으로 찾습니다. 메서드는 변수에 저장해 재사용할 수도 있습니다.

In [ ]:
class Warehouse:
    purpose = "storage"
    region = "west"


first = Warehouse()
second = Warehouse()
second.region = "east"
print(first.purpose, first.region)
print(second.purpose, second.region)


class Greeter:
    def greet(self, name):
        return f"안녕하세요, {name}님"


greeter = Greeter()
method = greeter.greet
print(method("학습자"))

## 8. 상속

상속을 사용하면 기존 기능을 재사용하면서 새 기능을 덧붙일 수 있습니다.

In [ ]:
class Asset:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self.price = price

    def describe(self):
        return f"{self.symbol}: {self.price:,}원"


class CoinAsset(Asset):
    def change_price(self, rate):
        self.price *= 1 + rate


bitcoin = CoinAsset("BTC", 100_000_000)
bitcoin.change_price(0.05)
print(bitcoin.describe())
print(isinstance(bitcoin, CoinAsset))
print(isinstance(bitcoin, Asset))
print(issubclass(CoinAsset, Asset))

## 9. 메서드 재정의와 `super()`

하위 클래스는 부모 메서드를 다시 정의할 수 있고, 부모 초기화를 재사용할 수도 있습니다.

In [ ]:
class Order:
    def summary(self):
        return "일반 주문"


class LimitOrder(Order):
    def __init__(self, limit_price):
        self.limit_price = limit_price

    def summary(self):
        return f"지정가 주문: {self.limit_price:,}원"


class MarketOrder(Order):
    def __init__(self, symbol, amount, slippage=0.001):
        super().__init__()
        self.symbol = symbol
        self.amount = amount
        self.slippage = slippage


order = LimitOrder(100_000_000)
print(order.summary())
market_order = MarketOrder("BTC", 10_000)
print(market_order.symbol, market_order.amount, market_order.slippage)

## 10. 다중 상속

Python은 여러 부모 클래스를 상속받는 구조를 지원합니다.

In [ ]:
class PriceFeed:
    def fetch_price(self):
        return 105_000_000


class RiskChecker:
    def check_risk(self, amount):
        return amount <= 1_000_000


class TradingBot(PriceFeed, RiskChecker):
    def can_trade(self, amount):
        return self.check_risk(amount) and self.fetch_price() > 0


bot = TradingBot()
print(bot.fetch_price())
print(bot.can_trade(500_000))
print(TradingBot.__mro__)

## 11. 비공개 이름과 이름 장식

`_`는 내부용 관례이고, `__`는 이름 장식을 적용해 충돌을 줄입니다.

In [ ]:
class Account:
    def __init__(self, balance):
        self._balance = balance

    def get_balance(self):
        return self._balance


account = Account(100_000)
print(account.get_balance())
print(account._balance)


class SecureAccount:
    def __init__(self, balance):
        self.__balance = balance

    def get_balance(self):
        return self.__balance


secure = SecureAccount(100_000)
print(secure.get_balance())
print(secure.__dict__)

## 12. 데이터클래스

`dataclass`를 사용하면 반복적인 초기화와 표현 코드 없이 객체를 빠르게 만들 수 있습니다.

In [ ]:
from dataclasses import dataclass


@dataclass
class MarketPrice:
    symbol: str
    price: float
    change_rate: float = 0.0


market_price = MarketPrice("BTC", 105_000_000, 2.35)
print(market_price)
print(market_price.symbol)
print(MarketPrice("ETH", 3_500_000) == MarketPrice("ETH", 3_500_000))

## 13. 반복자 프로토콜

`__iter__()`와 `__next__()`를 구현하면 직접 반복자를 만들 수 있습니다.

In [ ]:
class Reverse:
    def __init__(self, data):
        self.data = data
        self.index = len(data)

    def __iter__(self):
        return self

    def __next__(self):
        if self.index == 0:
            raise StopIteration
        self.index -= 1
        return self.data[self.index]


for symbol in Reverse(["BTC", "ETH", "XRP"]):
    print(symbol)

## 14. 제너레이터와 제너레이터 표현식

제너레이터는 `yield`를 사용해 필요한 순간에 값을 계산합니다.

In [ ]:
def reverse(data):
    for index in range(len(data) - 1, -1, -1):
        yield data[index]


for symbol in reverse(["BTC", "ETH", "XRP"]):
    print(symbol)


prices = [100, 105, 110, 98]
price_squares = (price ** 2 for price in prices)
print(sum(price_squares))

x_prices = [100, 105, 110]
y_prices = [2, 3, 4]
print(sum(x * y for x, y in zip(x_prices, y_prices)))

## 15. 실습 검증

챕터의 핵심 개념이 실제로 동작하는지 확인합니다.

In [ ]:
from pathlib import Path

workspace_dir = Path.cwd()
while workspace_dir != workspace_dir.parent and not (workspace_dir / "doc").exists():
    workspace_dir = workspace_dir.parent
source_path = workspace_dir / "doc" / "Chapter 11. 파이썬 클래스와 객체.md"
assert source_path.exists(), source_path
assert MarketPrice("BTC", 105_000_000, 2.35).symbol == "BTC"
assert list(Reverse(["BTC", "ETH"])) == ["ETH", "BTC"]
assert sum(x * y for x, y in zip([100, 105], [2, 3])) == 515
print("Chapter 11 실습 검증 통과")
print("원본 문서:", source_path)

## 실습 과제

1. `Coin` 클래스를 만들고 심볼, 가격, 변동률을 인스턴스 변수로 저장하세요.
2. `apply_change()` 메서드를 추가해 가격을 변동률만큼 변경하세요.
3. 모든 코인 객체가 공유하는 거래소 이름을 클래스 변수로 만들세요.
4. `Asset`과 `CoinAsset`을 상속 구조로 구성해 보세요.
5. `dataclass`로 주문 객체를 만들고 총 금액 계산 속성을 추가해 보세요.
6. `yield`를 이용해 가격 목록을 역순으로 생성하는 제너레이터를 작성하세요.